# 🚀 Hope ML: Professional Training Pipeline (Kaggle Edition)

This notebook provides a robust, professional environment for training the Hope Trading Transformer. Optimized for **Kaggle**.

### Key Features:
*   **Kaggle Native**: Loads `.env` directly from `/kaggle/working/`.\n*   **Path Agnostic**: Auto-detects Kaggle Input datasets vs live collection.
*   **Observability**: Real-time TensorBoard monitoring and data distribution plots.
*   **Integrity**: Automatic repo synchronization and dependency validation.

In [ ]:
# @title ## 1. Environment & Path Setup
import os
import sys
import subprocess

PROJECT_ROOT = "/kaggle/working/hope"
os.makedirs("/kaggle/working", exist_ok=True)

print(f"📂 Project Root: {PROJECT_ROOT}")

# Add to System Path
SCRIPTS_PATH = os.path.join(PROJECT_ROOT, "scripts")
if SCRIPTS_PATH not in sys.path:
    sys.path.insert(0, SCRIPTS_PATH)

print("✅ System path updated.")

In [ ]:
# @title ## 2. Repository Synchronization
REPO_URL = "https://github.com/planetazul3/hope.git" # @param {type:"string"}
BRANCH = "main" # @param {type:"string"}

def sync_repo():
    if not os.path.exists(os.path.join(PROJECT_ROOT, ".git")):
        print(f"Cloning {REPO_URL}...")
        subprocess.check_call(f"git clone --depth 1 {REPO_URL} {PROJECT_ROOT}", shell=True)
    else:
        print(f"Updating repository ({BRANCH})...")
        subprocess.check_call(f"git -C {PROJECT_ROOT} fetch origin", shell=True)
        subprocess.check_call(f"git -C {PROJECT_ROOT} reset --hard origin/{BRANCH}", shell=True)

try:
    sync_repo()
    os.chdir(PROJECT_ROOT)
    print(f"✨ Repository is up to date. Current dir: {os.getcwd()}")
except Exception as e:
    print(f"⚠️ Sync failed: {e}")

In [ ]:
# @title ## 3. Hardware & Dependency Audit (Transformer-Friendly)

print("📦 Installing modern dependency stack... (This takes a minute)")

!pip install -q "numpy>=2.1,<2.3"
!pip install -q transformers==4.49.0 datasets==3.3.2 accelerate==1.4.0 tokenizers==0.21.0 sentencepiece==0.2.0 evaluate==0.4.3 pandas==2.2.3 scikit-learn==1.6.1 tqdm==4.67.1 matplotlib==3.9.3 seaborn==0.13.2 tensorboard==2.19.0 onnx==1.17.0 onnxruntime-gpu==1.20.1

from IPython.display import clear_output
clear_output()

import platform
import sys
import os

print("✅ Dependencies successfully installed.\n")
print("🔍 Environment Audit:")
print(f"💻 OS: {platform.platform()}")
print(f"🐍 Python: {platform.python_version()}")

try:
    gpu_info = subprocess.check_output(
        "nvidia-smi --query-gpu=name,memory.total,utilization.gpu --format=csv,noheader", 
        shell=True
    ).decode().strip()
    print(f"🚀 GPU Active: {gpu_info}")
except Exception:
    print("❌ PyTorch/nvidia-smi does NOT detect a GPU. Training will be extremely slow.")

print("\n⚠️ CRITICAL STEP REQUIRED ⚠️")
print("If core C-libraries were updated, you must restart the Python engine.")
print("Go to: Run ➡️ Restart Session (or power button at top right)")

In [ ]:
# @title ## 4. Credentials & Environment Setup
# @markdown Upload your `.env` file to `/kaggle/working/` and it will be loaded automatically.

import os
import subprocess
import shutil

ENV_PATH = os.path.join(PROJECT_ROOT, ".env")
UPLOADED_ENV_PATH = "/kaggle/working/.env"
EXAMPLE_PATH = os.path.join(PROJECT_ROOT, ".env.example")

# 1. If user uploaded .env to /kaggle/working/, copy it into project root
if os.path.exists(UPLOADED_ENV_PATH) and os.path.abspath(UPLOADED_ENV_PATH) != os.path.abspath(ENV_PATH):
    print("📥 Found uploaded .env in /kaggle/working/. Copying to project root...")
    shutil.copy(UPLOADED_ENV_PATH, ENV_PATH)

# 2. Fallback to .env.example if no .env exists at all
if not os.path.exists(ENV_PATH) and os.path.exists(EXAMPLE_PATH):
    print("⚠️ No .env found. Falling back to .env.example...")
    subprocess.check_call(f"cp {EXAMPLE_PATH} {ENV_PATH}", shell=True)

# 3. Load into current session environment
if os.path.exists(ENV_PATH):
    with open(ENV_PATH, "r") as f:
        for line in f:
            if "=" in line and not line.startswith("#"):
                k, v = line.strip().split("=", 1)
                os.environ[k] = v.strip("\"\'")
    print("✅ Environment configuration synchronized.")
else:
    print("❌ ERROR: No .env configuration could be loaded.")

In [ ]:
# @title ## 5. Dataset Loading & Tick Collection

import os
import sys
import subprocess
import sqlite3
import pandas as pd

PROJECT_ROOT = "/kaggle/working/hope"
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
ENV_PATH  = os.path.join(PROJECT_ROOT, ".env")
SCRIPTS   = os.path.join(PROJECT_ROOT, "scripts")

os.makedirs(DATA_DIR, exist_ok=True)

SYMBOL = None
if os.path.exists(ENV_PATH):
    with open(ENV_PATH) as f:
        for line in f:
            line = line.strip()
            if line.startswith("DERIV_SYMBOL="):
                SYMBOL = line.split("=", 1)[1].strip('\'" ')
                break

if not SYMBOL:
    raise RuntimeError("❌ DERIV_SYMBOL not found in .env")
print(f"📌 Target Symbol: {SYMBOL}")

DB_PATH  = os.path.join(DATA_DIR, "tick_store.db")
CSV_PATH = os.path.join(DATA_DIR, f"{SYMBOL}_ticks.csv")

# --- Try to find Dataset in Kaggle Input ---
KAGGLE_INPUT_DIR = "/kaggle/input"
found_csv = None
for root, dirs, files in os.walk(KAGGLE_INPUT_DIR):
    for file in files:
        if file.endswith("ticks.csv") and SYMBOL in file:
            found_csv = os.path.join(root, file)
            break
    if found_csv: break

if found_csv:
    print(f"✅ Found Kaggle Dataset: {found_csv}")
    if not os.path.exists(CSV_PATH):
        os.symlink(found_csv, CSV_PATH)
        print(f"🔗 Linked dataset to {CSV_PATH}")
else:
    print("⚠️ No Kaggle dataset found. Proceeding with live Tick Collection...")
    
    if os.path.exists(CSV_PATH) and os.path.getsize(CSV_PATH) > 0:
        try:
            first_row = pd.read_csv(CSV_PATH, header=None, nrows=1)
            if first_row.shape[1] >= 3:
                existing_sym = str(first_row.iloc[0, 0]).strip()
                if existing_sym not in (SYMBOL, "symbol"): 
                    print(f"⚠️ Symbol changed (found {existing_sym}). Wiping {CSV_PATH}.")
                    os.remove(CSV_PATH)
        except Exception as e:
            print(f"⚠️ Could not verify CSV symbol: {e}")

    def run_command(cmd, desc):
        print(f"▶ {desc}\n   {' '.join(cmd)}")
        res = subprocess.run(cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)
        if res.returncode != 0:
            print(f"❌ Failed:\n{res.stderr.strip()[-1000:]}")
            raise subprocess.CalledProcessError(res.returncode, cmd)
        print("   ✅ Done.")

    try:
        run_command([sys.executable, os.path.join(SCRIPTS, "tick_collector.py"), 
                     "--symbol", SYMBOL, "--db", DB_PATH, "--mode", "backfill"], 
                     "Fetching missing ticks")
    except Exception:
        print("⚠️ Collector offline. Proceeding with local DB data.")

    try:
        run_command([sys.executable, os.path.join(SCRIPTS, "export_db.py"), 
                     "--db", DB_PATH, "--csv", CSV_PATH, "--symbol", SYMBOL, 
                     "--incremental", "--validate"], 
                     "Exporting to CSV")
    except Exception:
        print("⚠️ Incremental export failed. Using chunked SQLite fallback to prevent OOM...")
        conn = sqlite3.connect(DB_PATH)
        chunks = pd.read_sql_query(f"SELECT symbol, epoch, quote FROM ticks WHERE symbol = '{SYMBOL}' ORDER BY epoch", conn, chunksize=500000)
        first_chunk = True
        for chunk in chunks:
            chunk.to_csv(CSV_PATH, mode='w' if first_chunk else 'a', index=False, header=False)
            first_chunk = False
        conn.close()
        print("   ✅ Fallback export complete.")

print("📊 Loading dataset into memory for analysis...")
df_ticks = pd.read_csv(CSV_PATH, header=None, names=["symbol", "epoch", "quote"], on_bad_lines='skip')

if pd.isna(pd.to_numeric(df_ticks['quote'], errors='coerce')).all():
    df_ticks = pd.read_csv(CSV_PATH, header=None, names=["epoch", "quote"], on_bad_lines='skip')

df_ticks['epoch'] = pd.to_numeric(df_ticks['epoch'], errors='coerce')
df_ticks['quote'] = pd.to_numeric(df_ticks['quote'], errors='coerce')
df_ticks.dropna(subset=['epoch', 'quote'], inplace=True)

if df_ticks.empty or (df_ticks["quote"] <= 0).any():
    raise RuntimeError("❌ CSV contains invalid or zero-price data.")

print(f"✅ Ready: {len(df_ticks):,} ticks loaded.")

In [ ]:
# @title ## 6. Data Integrity & Distribution Analysis

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

if 'df_ticks' not in locals() or df_ticks.empty:
    raise RuntimeError("❌ df_ticks not found in memory. Run Cell 5 first.")

print(f"📈 Analyzing {len(df_ticks):,} ticks for {SYMBOL}...")

df_clean = df_ticks.drop_duplicates(subset=['epoch']).copy()
dropped = len(df_ticks) - len(df_clean)
if dropped > 0:
    print(f"⚠️ Removed {dropped:,} duplicate epoch entries.")

sns.set_theme(style="darkgrid")
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

tail_data = df_clean.tail(5000)
axes[0].plot(tail_data["epoch"], tail_data["quote"], color='blue', linewidth=1.2)
axes[0].set_title("Recent Price Action (Last 5k Ticks)", fontweight='bold')
axes[0].set_xlabel("Epoch Timestamp")
axes[0].set_ylabel("Quote Price")

returns = np.log(df_clean["quote"] / df_clean["quote"].shift(1)).dropna()

sns.histplot(returns, kde=True, bins=150, ax=axes[1], color='purple')
axes[1].set_title("Tick-to-Tick Volatility (Log Returns)", fontweight='bold')
axes[1].set_xlabel("Log Return")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

print("\n📊 Return Distribution Stats:")
print(f"Mean: {returns.mean():.6f} | Std Dev: {returns.std():.6f}")
print(f"Skew: {returns.skew():.4f}  | Kurtosis: {returns.kurtosis():.4f}")

In [ ]:
# @title ## 7. Training & Live Monitoring (Production Aligned)

%load_ext tensorboard

import os
import sys
import subprocess
import traceback
import gc

PROJECT_ROOT = "/kaggle/working/hope"
SCRIPTS_DIR  = os.path.join(PROJECT_ROOT, "scripts")
LOG_DIR      = os.path.join(PROJECT_ROOT, "logs")
ENV_PATH     = os.path.join(PROJECT_ROOT, ".env")

os.makedirs(LOG_DIR, exist_ok=True)

if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)

os.chdir(PROJECT_ROOT)

SYMBOL = None
if os.path.exists(ENV_PATH):
    with open(ENV_PATH) as f:
        for line in f:
            line = line.strip()
            if line.startswith("DERIV_SYMBOL="):
                SYMBOL = line.split("=", 1)[1].strip('"\' ')
                break

if not SYMBOL:
    SYMBOL = os.environ.get("DERIV_SYMBOL")

if not SYMBOL:
    raise RuntimeError("❌ DERIV_SYMBOL not found in .env. Set it and run Cell 5.")

os.environ["DERIV_SYMBOL"] = SYMBOL
print(f"📌 Training Target: {SYMBOL}")

CSV_PATH = os.path.join(PROJECT_ROOT, "data", f"{SYMBOL}_ticks.csv")
if not os.path.exists(CSV_PATH) or os.path.getsize(CSV_PATH) == 0:
    raise FileNotFoundError(f"❌ Tick data missing or empty at {CSV_PATH}. Run Cell 5.")
print(f"📂 Dataset verified: {CSV_PATH}")

import torch
if torch.cuda.is_available():
    print(f"🚀 Training on GPU: {torch.cuda.get_device_name(0)}")
    torch.cuda.empty_cache()
else:
    print("⚠️ WARNING: GPU NOT detected. Training will be extremely slow.")

print(f"📈 Launching TensorBoard logs → {LOG_DIR}")
os.system("pkill -f tensorboard")
%tensorboard --logdir {LOG_DIR}

print("\n🏗️ Starting ML pipeline (Phase 1: TS2Vec, Phase 2: Supervised MTL)")
print("   Architecture: Canonical Causal Transformer (L=32, 8 features)")

try:
    import train_fixed
    
    train_fixed.main(csv_path=CSV_PATH, log_dir=LOG_DIR)
    print("\n✅ Training finished successfully.")

    print("\n📦 Verifying model artifacts in Project Root...")
    artifacts = ["model.onnx", "model.onnx.sig", "best_model.pth"]
    saved_count = 0
    
    for fname in artifacts:
        fpath = os.path.join(PROJECT_ROOT, fname)
        if os.path.exists(fpath):
            size_mb = os.path.getsize(fpath) / 1e6
            print(f"   • Verified: {fname} ({size_mb:.2f} MB)")
            if os.path.abspath(fpath) != os.path.abspath(os.path.join("/kaggle/working/", fname)):
                os.system(f"cp {fpath} /kaggle/working/")
            saved_count += 1
            
    if saved_count == 0:
        print("   ⚠️ No output models found in project root. Did train_fixed fail to export?")
    elif saved_count < len(artifacts):
        print("   ⚠️ Some artifacts are missing. Check logs for quantization or signing errors.")
        
    print("\n✅ Models available in /kaggle/working/ for download.")

except ModuleNotFoundError:
    print("❌ Module `train_fixed` not found in `scripts/`.")
except Exception:
    print("❌ Training failed. Traceback:")
    traceback.print_exc()
finally:
    if 'train_fixed' in sys.modules:
        del sys.modules['train_fixed']
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()